In [0]:
import requests, json, time
from datetime import datetime, timedelta, timezone

HOURS_BACK = 24
BRONZE = "workspace.delhi_air.bronze_openaq_measurements"

key = dbutils.secrets.get(scope="openaq", key="api_key")
headers = {"X-API-Key": key}

sensor_ids = [r.sensor_id for r in spark.sql(
    "SELECT DISTINCT sensor_id FROM workspace.delhi_air.silver_pm25_hourly"
).collect()]
print("sensors to fetch:", len(sensor_ids))

now_utc = datetime.now(timezone.utc)
start = now_utc - timedelta(hours=HOURS_BACK)
fmt = "%Y-%m-%dT%H:%M:%SZ"

rows, failed = [], []
for sid in sensor_ids:
    r = requests.get(
        f"https://api.openaq.org/v3/sensors/{sid}/measurements",
        headers=headers,
        params={"datetime_from": start.strftime(fmt), "datetime_to": now_utc.strftime(fmt), "limit": 1000},
        timeout=30,
    )
    if r.status_code == 200:
        for item in r.json()["results"]:
            rows.append((sid, now_utc, json.dumps(item)))
    else:
        failed.append((sid, r.status_code))
        if r.status_code == 429:
            break
    time.sleep(1.2)

print("rows:", len(rows), "failed:", failed)

if rows:
    df = spark.createDataFrame(rows, schema="sensor_id long, ingested_at timestamp, raw_json string")
    df.write.mode("append").saveAsTable(BRONZE)
    print("saved")

if failed:
    raise RuntimeError(f"{len(failed)} sensor requests failed: {failed[:5]}")